# Causal AI System for Home Diagnosis of COVID-19 Using DEMI Algorithm

## Introduction & Objective

The objective of this project is to develop a causal AI system using the DEMI algorithm to estimate the probability of COVID-19 diagnosis based on information available at home prior to a clinic or emergency room visit.

This project uses temporal ordering, co-occurrence patterns, and real patient symptom data to support home-based COVID-19 risk prediction before laboratory confirmation.

## Data Source

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path

# File paths
patient_data_path = "/Users/natalie/Downloads/Final Project HAP 823/data/COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv"
knowledgebase_path = "/Users/natalie/Downloads/Final Project HAP 823/data/COVIDCARE_DEMI_knowledgebase_v4.csv"

In [2]:
# Load patient dataset and DEMI knowledgebase
patient_data = pd.read_csv(patient_data_path)
knowledgebase = pd.read_csv(knowledgebase_path)

# Clean column names by removing extra spaces
patient_data.columns = [col.strip() for col in patient_data.columns]
knowledgebase.columns = [col.strip() for col in knowledgebase.columns]

# Confirm files loaded correctly
print("Patient data shape:", patient_data.shape)
print("Knowledgebase shape:", knowledgebase.shape)

Patient data shape: (822, 472)
Knowledgebase shape: (70983, 11)


In [3]:
# Preview patient data
patient_data.head()

,Internal ID,Cohort,PCR Test Positive,PCR Test Date_DEID,Consent Submission Date_DEID,29789-consent_18yrs,29790-consent_applies,32985-consent_applies_cohort4,29791-consent_english,29792-consent_covid+,...,32353-pinkline_results_2,32356-pinkblue_confirm_2,32359-blue_nopink_confirm_2,32362-noblue_confirm_2,32364-unable_select_2,32365-unable_other_2,32366-which_test_2,32367-willing_retake_2,32373-invalid_followup_2,32374-test1_invalid_2
0,2994,3,0.0,7.0,0,1,3,0,1,2,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2995,3,0.0,7.0,0,1,3,0,1,2,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2996,1,0.0,7.0,0,1,3,0,1,2,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2997,2,0.0,7.0,0,1,3,0,1,2,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2998,3,0.0,7.0,0,1,3,0,1,2,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Review column names
print("Patient data first 20 columns:")
print(patient_data.columns[:20].tolist())

print("\nKnowledgebase columns:")
print(knowledgebase.columns.tolist())

Patient data first 20 columns:
['Internal ID', 'Cohort', 'PCR Test Positive', 'PCR Test Date_DEID', 'Consent Submission Date_DEID', '29789-consent_18yrs', '29790-consent_applies', '32985-consent_applies_cohort4', '29791-consent_english', '29792-consent_covid+', '29793-consent_location', '29794-consent_noeligble', '32795-future_researchName', '32797-future_researchPhone', '32799-future_researchEmail', '34683-consent_invited', '34684-consent_travel', '29826-storage_permission', '29829-consent', 'Shipment Confirmation Submission Date_DEID']

Knowledgebase columns:
['row_order', 'requested_concept_code', 'target_concept_code', 'concept_code', 'standard_concept_name', 'n_code_target', 'n_code_no_target', 'n_target_no_code', 'n_no_code_no_target', 'n_target_before_code', 'n_code_before_target']


In [5]:
# Check outcome variable
outcome_variable = "PCR Test Positive"

print("Outcome variable:", outcome_variable)
print("\nPCR Test Positive counts:")
print(patient_data[outcome_variable].value_counts(dropna=False))

Outcome variable: PCR Test Positive

PCR Test Positive counts:
PCR Test Positive
0.0    501
NaN    263
1.0     58
Name: count, dtype: int64


In [6]:
# Preview DEMI knowledgebase
knowledgebase.head()

,row_order,requested_concept_code,target_concept_code,concept_code,standard_concept_name,n_code_target,n_code_no_target,n_target_no_code,n_no_code_no_target,n_target_before_code,n_code_before_target
0,1,30082-Race-2,30082-Race-2,30082-Race-1,Race selected; White,20,385,333,45,333,385
1,2,30082-Race-3,30082-Race-3,30082-Race-1,Race selected; White,3,402,27,351,27,402
2,3,30082-Race-4,30082-Race-4,30082-Race-1,Race selected; White,7,398,6,372,6,398
3,4,30082-Race-5,30082-Race-5,30082-Race-1,Race selected; White,0,405,2,376,2,405
4,5,30082-Race-6,30082-Race-6,30082-Race-1,Race selected; White,2,403,11,367,11,403


In [7]:
# Create analytic dataset with known PCR test outcome
analytic_data = patient_data.dropna(subset=[outcome_variable]).copy()

# Convert outcome to integer
analytic_data[outcome_variable] = analytic_data[outcome_variable].astype(int)

print("Analytic dataset shape:", analytic_data.shape)
print("\nPCR Test Positive counts in analytic dataset:")
print(analytic_data[outcome_variable].value_counts())

Analytic dataset shape: (559, 472)

PCR Test Positive counts in analytic dataset:
PCR Test Positive
0    501
1     58
Name: count, dtype: int64


The project uses the COVIDCARE patient dataset and the DEMI knowledgebase file provided for the final project. The patient dataset contains 822 records and 472 variables, including survey responses, symptoms, home testing variables, and PCR test results. The DEMI knowledgebase contains 70,983 rows and 11 columns and is used by the DEMI algorithm to analyze relationships between variables.

The main outcome variable is PCR Test Positive. There are 501 negative PCR cases, 58 positive PCR cases, and 263 missing PCR results. For analyses that require a known outcome, I created an analytic dataset with the 559 records that have non-missing PCR results.

In [8]:
print(patient_data.shape)
print(knowledgebase.shape)
print(analytic_data.shape)

(822, 472)
(70983, 11)
(559, 472)


## Temporal Ordering

The DEMI algorithm needs the variables to be in a logical order. For this project, I grouped the variables based on when they would happen in real life:

- Tier 0: Demographics
- Tier 1: Vaccination and prior health history
- Tier 2: Symptoms, exposures, behaviors, and conditions
- Tier 3: At-home rapid test results
- Tier 4: PCR Test Positive

The model should only use information that would be available before the final PCR test result.

## Knowledge Base Construction

In [9]:
# Preview the DEMI knowledgebase
print("Knowledgebase shape:", knowledgebase.shape)
display(knowledgebase.head())

print("\nKnowledgebase columns:")
print(knowledgebase.columns.tolist())

Knowledgebase shape: (70983, 11)


,row_order,requested_concept_code,target_concept_code,concept_code,standard_concept_name,n_code_target,n_code_no_target,n_target_no_code,n_no_code_no_target,n_target_before_code,n_code_before_target
0,1,30082-Race-2,30082-Race-2,30082-Race-1,Race selected; White,20,385,333,45,333,385
1,2,30082-Race-3,30082-Race-3,30082-Race-1,Race selected; White,3,402,27,351,27,402
2,3,30082-Race-4,30082-Race-4,30082-Race-1,Race selected; White,7,398,6,372,6,398
3,4,30082-Race-5,30082-Race-5,30082-Race-1,Race selected; White,0,405,2,376,2,405
4,5,30082-Race-6,30082-Race-6,30082-Race-1,Race selected; White,2,403,11,367,11,403



Knowledgebase columns:
['row_order', 'requested_concept_code', 'target_concept_code', 'concept_code', 'standard_concept_name', 'n_code_target', 'n_code_no_target', 'n_target_no_code', 'n_no_code_no_target', 'n_target_before_code', 'n_code_before_target']


The knowledge base shows relationships between a variable and a target variable. It includes counts for how often variables occur together, how often one occurs without the other, and which variable is recorded as occurring first. These counts are used by the DEMI algorithm to estimate relationships between symptoms, testing variables, and COVID-19 diagnosis.

## DEMI Algorithm

In [10]:
from pathlib import Path
import json

reference_notebook_path = Path("/Users/natalie/Downloads/Final Project HAP 823/data/Covid _Prediction.ipynb")

with open(reference_notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Number of cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"][:15]):
    cell_type = cell["cell_type"]
    source = "".join(cell["source"])
    print(f"\n--- Cell {i} ({cell_type}) ---")
    print(source[:1000])

Number of cells: 29

--- Cell 0 (code) ---
# =========================================================
# QUESTION 1 - COMPLETE COVID AI SYSTEM CODE
# =========================================================
# Files used:
#   1) Raw patient-level file
#   2) DEMI knowledgebase file
#   3) Survey dictionary (optional for labels)
#
# This script does:
#   - Load all files
#   - Create tier assignments
#   - Build knowledgebase logic
#   - Calculate pairwise associations
#   - Frequency of co-occurrence
#   - Logistic / LASSO / Boosting models
#   - McFadden pseudo R-square
#   - Direct predictors of PCR
#   - Parent regressions for Markov blanket
#   - Clean network plot
#   - CPT tables for Netica
#   - Final COVID probability prediction
# =========================================================

import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection i

In [11]:
# =========================================================
# DEMI Algorithm Setup
# =========================================================

import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# File paths for this project
RAW_FILE = "/Users/natalie/Downloads/Final Project HAP 823/data/COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv"
KB_FILE = "/Users/natalie/Downloads/Final Project HAP 823/data/COVIDCARE_DEMI_knowledgebase_v4.csv"

# Pull from loaded datasets
df = patient_data.copy()
kb = knowledgebase.copy()

TARGET = "PCR Test Positive"

print("Raw patient data shape:", df.shape)
print("Knowledgebase shape:", kb.shape)
print("Target variable:", TARGET)

Raw patient data shape: (822, 472)
Knowledgebase shape: (70983, 11)
Target variable: PCR Test Positive


In [12]:
# ---------------------------------------------------------
# Create tier assignment function
# ---------------------------------------------------------

def assign_tier(col_name: str) -> int:
    c = str(col_name).lower()

    # Tier 4: PCR lab confirmation
    if c == "pcr test positive":
        return 4

    # Tier 3: At-home testing variables
    if (
        "pinkline" in c
        or "blueline" in c
        or "pinkblue_confirm" in c
        or "blue_nopink_confirm" in c
        or "noblue_confirm" in c
        or "athome" in c
        or "testkit_performing" in c
        or "which_test" in c
    ):
        return 3

    # Tier 1: Vaccination variables
    if (
        "vaccine" in c
        or "vacc" in c
        or "flu_shot" in c
        or "covid_vaccine" in c
    ):
        return 1

    # Tier 0: Demographics / baseline variables
    if (
        "dob" in c
        or "age" in c
        or "gender" in c
        or "race" in c
        or "ethnic" in c
        or "sex" in c
    ):
        return 0

    # Tier 2: Symptoms, exposures, behaviors, and conditions
    return 2


# Create a tier map for all patient data columns
tier_map = {col: assign_tier(col) for col in df.columns}

# Show how many variables are in each tier
tier_counts = pd.Series(tier_map).value_counts().sort_index()

print("Number of variables by tier:")
print(tier_counts)

Number of variables by tier:
0     27
1     50
2    377
3     17
4      1
Name: count, dtype: int64


I assigned each variable to a tier based on when it would occur in real life. Demographic variables come first, followed by vaccination or prior history, symptoms and exposures, at-home testing, and finally the PCR test result.

In [13]:
# ---------------------------------------------------------
# Remove self-comparisons and add tier information
# ---------------------------------------------------------

# Remove rows where the variable is being compared to itself
kb = kb[kb["concept_code"] != kb["target_concept_code"]].copy()

# Add tier information to the knowledgebase
kb["concept_tier"] = kb["concept_code"].map(tier_map)
kb["target_tier"] = kb["target_concept_code"].map(tier_map)

# If a variable is not found in the patient data column names, assign default Tier 2
kb["concept_tier"] = kb["concept_tier"].fillna(2).astype(int)
kb["target_tier"] = kb["target_tier"].fillna(2).astype(int)

print("Knowledgebase after removing self-comparisons:", kb.shape)

print("\nConcept tier counts:")
print(kb["concept_tier"].value_counts().sort_index())

print("\nTarget tier counts:")
print(kb["target_tier"].value_counts().sort_index())

kb.head()

Knowledgebase after removing self-comparisons: (70983, 13)

Concept tier counts:
concept_tier
0     5643
1    10526
2    54814
Name: count, dtype: int64

Target tier counts:
target_tier
0      361
1     2318
2    68304
Name: count, dtype: int64


,row_order,requested_concept_code,target_concept_code,concept_code,standard_concept_name,n_code_target,n_code_no_target,n_target_no_code,n_no_code_no_target,n_target_before_code,n_code_before_target,concept_tier,target_tier
0,1,30082-Race-2,30082-Race-2,30082-Race-1,Race selected; White,20,385,333,45,333,385,0,0
1,2,30082-Race-3,30082-Race-3,30082-Race-1,Race selected; White,3,402,27,351,27,402,0,0
2,3,30082-Race-4,30082-Race-4,30082-Race-1,Race selected; White,7,398,6,372,6,398,0,0
3,4,30082-Race-5,30082-Race-5,30082-Race-1,Race selected; White,0,405,2,376,2,405,0,0
4,5,30082-Race-6,30082-Race-6,30082-Race-1,Race selected; White,2,403,11,367,11,403,0,0


In [14]:
# ---------------------------------------------------------
# Apply temporal rules
# ---------------------------------------------------------

def apply_temporal_rules(row):
    n11 = row["n_code_target"]
    n10 = row["n_code_no_target"]
    n01 = row["n_target_no_code"]

    ct = row["concept_tier"]
    tt = row["target_tier"]

    # Rule 1: If the two variables never occur together
    if n11 == 0:
        row["n_code_before_target_final"] = n10
        row["n_target_before_code_final"] = n01
        return row

    # Rule 2: If concept is in an earlier tier, it occurs before target
    if ct < tt:
        row["n_code_before_target_final"] = n11
        row["n_target_before_code_final"] = 0
        return row

    # Rule 3: If concept is in a later tier, target occurs before concept
    if ct > tt:
        row["n_code_before_target_final"] = 0
        row["n_target_before_code_final"] = n11
        return row

    # Rule 4: Same-tier variables use existing ordering information when available
    if pd.notnull(row.get("n_code_before_target", np.nan)) and pd.notnull(row.get("n_target_before_code", np.nan)):
        row["n_code_before_target_final"] = row["n_code_before_target"]
        row["n_target_before_code_final"] = row["n_target_before_code"]
        return row

    # Rule 5: If timing is unknown, split direction equally
    row["n_code_before_target_final"] = n11 / 2
    row["n_target_before_code_final"] = n11 / 2
    return row


kb = kb.apply(apply_temporal_rules, axis=1)

print("Temporal rules applied.")
print(kb[[
    "concept_code",
    "target_concept_code",
    "concept_tier",
    "target_tier",
    "n_code_target",
    "n_code_before_target_final",
    "n_target_before_code_final"
]].head())

Temporal rules applied.
   concept_code target_concept_code  concept_tier  target_tier  n_code_target  \
0  30082-Race-1        30082-Race-2             0            0             20   
1  30082-Race-1        30082-Race-3             0            0              3   
2  30082-Race-1        30082-Race-4             0            0              7   
3  30082-Race-1        30082-Race-5             0            0              0   
4  30082-Race-1        30082-Race-6             0            0              2   

   n_code_before_target_final  n_target_before_code_final  
0                         385                         333  
1                         402                          27  
2                         398                           6  
3                         405                           2  
4                         403                          11  


I applied temporal rules so that variables in earlier tiers are treated as occurring before variables in later tiers. When variables are in the same tier, the model uses the ordering information available in the knowledgebase.

In [15]:
# ---------------------------------------------------------
# Build 2x2 table components and calculate association measures
# ---------------------------------------------------------

kb["n11"] = kb["n_code_target"]
kb["n10"] = kb["n_code_no_target"]
kb["n01"] = kb["n_target_no_code"]
kb["n00"] = kb["n_no_code_no_target"]

kb["N"] = kb[["n11", "n10", "n01", "n00"]].sum(axis=1)

# Odds ratio with 0.5 correction to avoid division by zero
kb["odds_ratio"] = ((kb["n11"] + 0.5) * (kb["n00"] + 0.5)) / (
    (kb["n10"] + 0.5) * (kb["n01"] + 0.5)
)

kb["log_odds_ratio"] = np.log(kb["odds_ratio"])

# Phi coefficient
num = kb["n11"] * kb["n00"] - kb["n10"] * kb["n01"]
den = np.sqrt(
    (kb["n11"] + kb["n10"])
    * (kb["n01"] + kb["n00"])
    * (kb["n11"] + kb["n01"])
    * (kb["n10"] + kb["n00"])
)

kb["phi"] = np.where(den == 0, np.nan, num / den)

# Conditional probabilities
kb["support_both"] = kb["n11"] / kb["N"]

kb["p_target_given_code"] = np.where(
    (kb["n11"] + kb["n10"]) == 0,
    np.nan,
    kb["n11"] / (kb["n11"] + kb["n10"])
)

kb["p_target_given_no_code"] = np.where(
    (kb["n01"] + kb["n00"]) == 0,
    np.nan,
    kb["n01"] / (kb["n01"] + kb["n00"])
)

pairwise_assoc = kb[
    [
        "concept_code",
        "target_concept_code",
        "concept_tier",
        "target_tier",
        "n11",
        "n10",
        "n01",
        "n00",
        "odds_ratio",
        "log_odds_ratio",
        "phi",
        "support_both",
        "p_target_given_code",
        "p_target_given_no_code",
    ]
].copy()

print("Pairwise association table shape:", pairwise_assoc.shape)

pairwise_assoc.sort_values("odds_ratio", ascending=False).head(10)

Pairwise association table shape: (70983, 14)


,concept_code,target_concept_code,concept_tier,target_tier,n11,n10,n01,n00,odds_ratio,log_odds_ratio,phi,support_both,p_target_given_code,p_target_given_no_code
19423,30150-flu_tst_symptoms-6,30147-flu_why-1,2,2,8,0,0,718,24429.0,10.103526,1.000000,0.011019,1.000000,0.00000
16146,30147-flu_why-1,30150-flu_tst_symptoms-6,2,2,8,0,0,718,24429.0,10.103526,1.000000,0.011019,1.000000,0.00000
27702,30158-Symtpom_Neuro-999,30160-Symptom_Inflamm-999,2,2,3,0,0,723,10129.0,9.223158,1.000000,0.004132,1.000000,0.00000
29107,30160-Symptom_Inflamm-999,30158-Symtpom_Neuro-999,2,2,3,0,0,723,10129.0,9.223158,1.000000,0.004132,1.000000,0.00000
69885,bin_30171-COVID_vaccine,bin_32136-vaccine_didyou,2,2,577,3,2,123,8151.0,9.005896,0.975777,0.818440,0.994828,0.01600
70396,bin_32136-vaccine_didyou,bin_30171-COVID_vaccine,2,2,577,2,3,123,8151.0,9.005896,0.975777,0.818440,0.996546,0.02381
60483,32726-Gender_more-1,30086-Gender-5,0,0,1,0,0,782,4695.0,8.454253,1.000000,0.001277,1.000000,0.00000
3523,30086-Gender-5,32726-Gender_more-1,0,0,1,0,0,782,4695.0,8.454253,1.000000,0.001277,1.000000,0.00000
18963,30150-flu_tst_symptoms-30,30150-flu_tst_symptoms-23,2,2,1,0,0,725,4353.0,8.378621,1.000000,0.001377,1.000000,0.00000
31239,30183-states_specify-20,30183-states_specify-24,2,2,1,0,0,725,4353.0,8.378621,1.000000,0.001377,1.000000,0.00000


In [16]:
# ---------------------------------------------------------
# Clean up association table
# ---------------------------------------------------------

# Add total count if it is not already in pairwise_assoc
pairwise_assoc["N"] = pairwise_assoc[["n11", "n10", "n01", "n00"]].sum(axis=1)

# Filter out very rare pairings to avoid misleading large odds ratios
pairwise_assoc_clean = pairwise_assoc[
    (pairwise_assoc["n11"] >= 10) &
    (pairwise_assoc["N"] > 0)
].copy()

# Sort by absolute phi coefficient instead of odds ratio
pairwise_assoc_clean = pairwise_assoc_clean.reindex(
    pairwise_assoc_clean["phi"].abs().sort_values(ascending=False).index
)

print("Cleaned pairwise association table shape:", pairwise_assoc_clean.shape)

pairwise_assoc_clean.head(15)

Cleaned pairwise association table shape: (8845, 15)


,concept_code,target_concept_code,concept_tier,target_tier,n11,n10,n01,n00,odds_ratio,log_odds_ratio,phi,support_both,p_target_given_code,p_target_given_no_code,N
70396,bin_32136-vaccine_didyou,bin_30171-COVID_vaccine,2,2,577,2,3,123,8151.000000,9.005896,0.975777,0.818440,0.996546,0.023810,705
69885,bin_30171-COVID_vaccine,bin_32136-vaccine_didyou,2,2,577,3,2,123,8151.000000,9.005896,0.975777,0.818440,0.994828,0.016000,705
69886,bin_30171-COVID_vaccine,bin_36096-Post_covidvaccine,2,2,231,1,4,65,2246.407407,7.717088,0.952654,0.767442,0.995690,0.057971,301
70673,bin_36096-Post_covidvaccine,bin_30171-COVID_vaccine,2,2,231,4,1,65,2246.407407,7.717088,0.952654,0.767442,0.982979,0.015152,301
70398,bin_32136-vaccine_didyou,bin_36096-Post_covidvaccine,2,2,231,1,5,62,1753.787879,7.469533,0.941769,0.772575,0.995690,0.074627,299
70675,bin_36096-Post_covidvaccine,bin_32136-vaccine_didyou,2,2,231,5,1,62,1753.787879,7.469533,0.941769,0.772575,0.978814,0.015873,299
297,30082-Race-2,30082-Race-1,0,0,20,333,385,45,0.007255,-4.926047,-0.835123,0.025543,0.056657,0.895349,783
0,30082-Race-1,30082-Race-2,0,0,20,385,333,45,0.007255,-4.926047,-0.835123,0.025543,0.049383,0.880952,783
11177,30141-covid_tst_symptoms-15,30141-covid_tst_symptoms-16,2,2,14,1,5,706,1241.727273,7.124259,0.825365,0.019284,0.933333,0.007032,726
11412,30141-covid_tst_symptoms-16,30141-covid_tst_symptoms-15,2,2,14,5,1,706,1241.727273,7.124259,0.825365,0.019284,0.736842,0.001414,726


Because very rare variable pairs can produce very large odds ratios, I also created a cleaner association table that only includes pairs occurring together at least 10 times. I sorted this table by the absolute value of the phi coefficient to focus on stronger and more stable relationships.

In [17]:
# ---------------------------------------------------------
# Find all knowledgebase rows connected to PCR Test Positive
# ---------------------------------------------------------

# Search for PCR-related rows in either concept_code or target_concept_code
pcr_rows = pairwise_assoc[
    pairwise_assoc["concept_code"].astype(str).str.lower().str.contains("pcr|test positive", na=False) |
    pairwise_assoc["target_concept_code"].astype(str).str.lower().str.contains("pcr|test positive", na=False)
].copy()

print("PCR-related rows found:", pcr_rows.shape[0])

pcr_rows.head(20)

PCR-related rows found: 297


,concept_code,target_concept_code,concept_tier,target_tier,n11,n10,n01,n00,odds_ratio,log_odds_ratio,phi,support_both,p_target_given_code,p_target_given_no_code,N
296,30082-Race-1,target_pcr_positive,0,2,36,358,20,139,0.692826,-0.366977,-0.051633,0.065099,0.091371,0.125786,553
593,30082-Race-2,target_pcr_positive,0,2,19,105,37,392,1.934597,0.659899,0.092599,0.034358,0.153226,0.086247,553
890,30082-Race-3,target_pcr_positive,0,2,1,29,55,468,0.429226,-0.845772,-0.053931,0.001808,0.033333,0.105163,553
1187,30082-Race-4,target_pcr_positive,0,2,0,12,56,485,0.343717,-1.067937,-0.049993,0.000000,0.000000,0.103512,553
1484,30082-Race-5,target_pcr_positive,0,2,0,2,56,495,1.753982,0.561889,-0.020223,0.000000,0.000000,0.101633,553
1781,30082-Race-6,target_pcr_positive,0,2,1,12,55,485,1.049730,0.048533,-0.012520,0.001808,0.076923,0.101852,553
2078,30082-Race-998,target_pcr_positive,0,2,0,4,56,493,0.970501,-0.029942,-0.028652,0.000000,0.000000,0.102004,553
2375,30082-Race-999,target_pcr_positive,0,2,0,3,56,494,1.250316,0.223396,-0.024791,0.000000,0.000000,0.101818,553
2672,30086-Gender-1,target_pcr_positive,0,2,25,148,31,349,1.905243,0.644610,0.096716,0.045208,0.144509,0.081579,553
2969,30086-Gender-2,target_pcr_positive,0,2,31,347,25,150,0.534998,-0.625492,-0.093806,0.056058,0.082011,0.142857,553


In [18]:
# ---------------------------------------------------------
# Sort PCR-related rows by strongest association
# ---------------------------------------------------------

pcr_rows_sorted = pcr_rows.reindex(
    pcr_rows["phi"].abs().sort_values(ascending=False).index
)

pcr_rows_sorted.head(15)

,concept_code,target_concept_code,concept_tier,target_tier,n11,n10,n01,n00,odds_ratio,log_odds_ratio,phi,support_both,p_target_given_code,p_target_given_no_code,N
42408,31386-covid_results-1,target_pcr_positive,2,2,19,11,19,430,37.434783,3.622600,0.529874,0.039666,0.633333,0.042316,479
27125,30158-Symtpom_Neuro-7,target_pcr_positive,2,2,24,11,28,463,34.647597,3.545228,0.524951,0.045627,0.685714,0.057026,526
70982,target_athome_positive,target_pcr_positive,2,2,23,11,27,449,33.401581,3.508603,0.519869,0.045098,0.676471,0.056723,510
27360,30158-Symtpom_Neuro-8,target_pcr_positive,2,2,20,8,32,466,34.618100,3.544377,0.488917,0.038023,0.714286,0.064257,526
15140,30141-covid_tst_symptoms-6,target_pcr_positive,2,2,31,36,21,438,17.601465,2.867982,0.465716,0.058935,0.462687,0.045752,526
42416,31386-covid_results-2,target_pcr_positive,2,2,22,430,16,11,0.036427,-3.312445,-0.464168,0.045929,0.048673,0.592593,479
21720,30153-Symptoms-6,target_pcr_positive,2,2,25,21,27,453,19.558985,2.973435,0.461149,0.047529,0.543478,0.056250,526
13965,30141-covid_tst_symptoms-3,target_pcr_positive,2,2,17,6,35,468,35.530878,3.570402,0.458711,0.032319,0.739130,0.069583,526
8560,30138-COVID_Why-1,target_pcr_positive,2,2,36,54,16,420,17.067834,2.837196,0.458398,0.068441,0.400000,0.036697,526
14905,30141-covid_tst_symptoms-5,target_pcr_positive,2,2,21,13,31,461,23.332745,3.149858,0.456921,0.039924,0.617647,0.063008,526


The knowledgebase uses target_pcr_positive as the label for the PCR outcome. The strongest PCR-related relationships included COVID test result variables, neurological symptom variables, at-home positive test results, and COVID symptom variables. This supports the goal of using home-based information, such as symptoms and at-home testing, to estimate the probability of PCR-confirmed COVID-19.

In [19]:
# ---------------------------------------------------------
# Prepare modeling data
# ---------------------------------------------------------

def is_date_or_id_or_admin(col):
    c = str(col).lower()
    if c == TARGET.lower():
        return False
    return (
        "date" in c
        or "submission" in c
        or "confirmation" in c
        or "internal id" in c
        or "cohort" in c
        or "_deid" in c
        or "name" in c
        or "phone" in c
        or "email" in c
    )

# Remove administrative/date/ID columns
usable_cols = [c for c in df.columns if not is_date_or_id_or_admin(c)]
model_df = df[usable_cols].copy()

# Keep target if it was removed
if TARGET not in model_df.columns:
    model_df[TARGET] = df[TARGET]

# Convert object columns to numeric category codes
for col in model_df.columns:
    if model_df[col].dtype == "object":
        model_df[col] = model_df[col].astype("category").cat.codes.replace(-1, np.nan)

# Convert boolean columns to integers
for col in model_df.columns:
    if str(model_df[col].dtype) == "bool":
        model_df[col] = model_df[col].astype(int)

# Ensure target is numeric and keep only known PCR outcomes
model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="coerce")
model_df = model_df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

# Keep only predictors that occur before PCR
predictor_cols = [c for c in model_df.columns if c != TARGET and assign_tier(c) < 4]

X_base = model_df[predictor_cols].copy()
y = model_df[TARGET].copy()

print("Modeling dataset shape:", model_df.shape)
print("Base predictor matrix shape:", X_base.shape)
print("Outcome counts:")
print(y.value_counts())

Modeling dataset shape: (559, 445)
Base predictor matrix shape: (559, 444)
Outcome counts:
PCR Test Positive
0    501
1     58
Name: count, dtype: int64


After removing administrative fields and records without PCR results, the modeling dataset included 559 records and 445 variables. The predictor matrix included 444 variables that occurred before the PCR outcome.

In [20]:
# ---------------------------------------------------------
# Create interaction terms
# ---------------------------------------------------------

# Focus interaction terms on symptom and home-testing variables
symptom_home_vars = [c for c in predictor_cols if assign_tier(c) in [2, 3]]

# Limit the number of variables used for interactions to keep the model manageable
pair_base = symptom_home_vars[:20]
triple_base = symptom_home_vars[:8]

X = X_base.copy()

# Pairwise interactions
for a, b in itertools.combinations(pair_base, 2):
    X[f"{a}__X__{b}"] = X[a].fillna(0) * X[b].fillna(0)

# Triple interactions
for a, b, c in itertools.combinations(triple_base, 3):
    X[f"{a}__X__{b}__X__{c}"] = X[a].fillna(0) * X[b].fillna(0) * X[c].fillna(0)

print("Number of symptom/home variables available:", len(symptom_home_vars))
print("Feature matrix after interactions:", X.shape)

Number of symptom/home variables available: 370
Feature matrix after interactions: (559, 690)


I created pairwise and three-way interaction terms from symptom and home-testing variables. This increased the feature matrix to 690 variables, allowing the model to consider combinations of symptoms and home-based information rather than only single variables.

In [21]:
# ---------------------------------------------------------
# Train/test split
# ---------------------------------------------------------

# Fill missing predictor values with 0 for modeling
X_model = X.fillna(0)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining outcome counts:")
print(y_train.value_counts())

print("\nTesting outcome counts:")
print(y_test.value_counts())

Training set shape: (391, 690)
Testing set shape: (168, 690)

Training outcome counts:
PCR Test Positive
0    350
1     41
Name: count, dtype: int64

Testing outcome counts:
PCR Test Positive
0    151
1     17
Name: count, dtype: int64


I split the modeling data into training and testing sets using a 70/30 split. I used stratification so the COVID-positive and COVID-negative cases were represented in both sets.

In [22]:
# ---------------------------------------------------------
# Train and evaluate prediction models
# ---------------------------------------------------------

results = []

# Model 1: Logistic Regression
log_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, log_pred),
    "ROC_AUC": roc_auc_score(y_test, log_prob),
    "Log_Loss": log_loss(y_test, log_prob)
})


# Model 2: LASSO Logistic Regression
lasso_model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(
        Cs=10,
        cv=5,
        penalty="l1",
        solver="liblinear",
        scoring="roc_auc",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

lasso_model.fit(X_train, y_train)

lasso_pred = lasso_model.predict(X_test)
lasso_prob = lasso_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "LASSO Logistic Regression",
    "Accuracy": accuracy_score(y_test, lasso_pred),
    "ROC_AUC": roc_auc_score(y_test, lasso_prob),
    "Log_Loss": log_loss(y_test, lasso_prob)
})


# Model 3: Gradient Boosting
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

results.append({
    "Model": "Gradient Boosting",
    "Accuracy": accuracy_score(y_test, gb_pred),
    "ROC_AUC": roc_auc_score(y_test, gb_prob),
    "Log_Loss": log_loss(y_test, gb_prob)
})


model_results = pd.DataFrame(results)

model_results

,Model,Accuracy,ROC_AUC,Log_Loss
0,Logistic Regression,0.916667,0.854694,1.414926
1,LASSO Logistic Regression,0.934524,0.867744,0.552635
2,Gradient Boosting,0.946429,0.947604,0.192881


I compared logistic regression, LASSO logistic regression, and gradient boosting models. Gradient boosting performed best overall, with the highest accuracy and ROC AUC and the lowest log loss. This suggests that a more flexible model was better able to capture patterns in the COVIDCARE symptom and home-testing data.

In [23]:
# Save model comparison results
model_results.to_csv("model_results.csv", index=False)

print("Saved model_results.csv")

Saved model_results.csv


## Interactive COVID Prediction

This section creates a simple notebook-based prediction tool. The user can select a patient record from the test set, and the model returns the predicted probability of PCR-confirmed COVID-19. This demonstrates how the trained model could support home-based COVID risk prediction using available symptom and testing information.

In [24]:
# ---------------------------------------------------------
# Interactive COVID Prediction
# ---------------------------------------------------------

def predict_covid_probability(patient_index=0):
    """
    Predicts the probability of PCR-confirmed COVID-19
    for one patient from the test dataset.
    """
    
    # Select one patient from the test set
    selected_patient = X_test.iloc[[patient_index]]
    
    # Predict probability using the best-performing model
    predicted_probability = gb_model.predict_proba(selected_patient)[0, 1]
    predicted_class = gb_model.predict(selected_patient)[0]
    actual_value = y_test.iloc[patient_index]
    
    print("Patient index from test set:", patient_index)
    print("Predicted probability of PCR-positive COVID-19:", round(predicted_probability, 3))
    print("Predicted class:", predicted_class)
    print("Actual PCR Test Positive value:", actual_value)
    
    if predicted_class == 1:
        print("Model interpretation: The model predicts this patient is COVID-positive.")
    else:
        print("Model interpretation: The model predicts this patient is COVID-negative.")

In [25]:
# Example prediction
predict_covid_probability(patient_index=0)

Patient index from test set: 0
Predicted probability of PCR-positive COVID-19: 0.001
Predicted class: 0
Actual PCR Test Positive value: 0
Model interpretation: The model predicts this patient is COVID-negative.


In [26]:
# Find COVID-positive cases in the test set
positive_test_indices = list(np.where(y_test.values == 1)[0])

print("COVID-positive test set indices:")
print(positive_test_indices[:10])

COVID-positive test set indices:
[np.int64(10), np.int64(11), np.int64(17), np.int64(25), np.int64(27), np.int64(28), np.int64(48), np.int64(108), np.int64(117), np.int64(136)]


In [27]:
predict_covid_probability(patient_index=10)

Patient index from test set: 10
Predicted probability of PCR-positive COVID-19: 0.943
Predicted class: 1
Actual PCR Test Positive value: 1
Model interpretation: The model predicts this patient is COVID-positive.


In [28]:
predict_covid_probability(patient_index=11)

Patient index from test set: 11
Predicted probability of PCR-positive COVID-19: 0.997
Predicted class: 1
Actual PCR Test Positive value: 1
Model interpretation: The model predicts this patient is COVID-positive.


I tested the prediction function using both negative and positive cases from the test set. The examples show that the function returns a predicted probability, predicted class, and actual PCR result for comparison.

## Results

In [29]:
# Display final model comparison results
model_results

,Model,Accuracy,ROC_AUC,Log_Loss
0,Logistic Regression,0.916667,0.854694,1.414926
1,LASSO Logistic Regression,0.934524,0.867744,0.552635
2,Gradient Boosting,0.946429,0.947604,0.192881


In [30]:
# Show strongest PCR-related relationships from the knowledgebase
pcr_rows_sorted.head(10)

,concept_code,target_concept_code,concept_tier,target_tier,n11,n10,n01,n00,odds_ratio,log_odds_ratio,phi,support_both,p_target_given_code,p_target_given_no_code,N
42408,31386-covid_results-1,target_pcr_positive,2,2,19,11,19,430,37.434783,3.622600,0.529874,0.039666,0.633333,0.042316,479
27125,30158-Symtpom_Neuro-7,target_pcr_positive,2,2,24,11,28,463,34.647597,3.545228,0.524951,0.045627,0.685714,0.057026,526
70982,target_athome_positive,target_pcr_positive,2,2,23,11,27,449,33.401581,3.508603,0.519869,0.045098,0.676471,0.056723,510
27360,30158-Symtpom_Neuro-8,target_pcr_positive,2,2,20,8,32,466,34.618100,3.544377,0.488917,0.038023,0.714286,0.064257,526
15140,30141-covid_tst_symptoms-6,target_pcr_positive,2,2,31,36,21,438,17.601465,2.867982,0.465716,0.058935,0.462687,0.045752,526
42416,31386-covid_results-2,target_pcr_positive,2,2,22,430,16,11,0.036427,-3.312445,-0.464168,0.045929,0.048673,0.592593,479
21720,30153-Symptoms-6,target_pcr_positive,2,2,25,21,27,453,19.558985,2.973435,0.461149,0.047529,0.543478,0.056250,526
13965,30141-covid_tst_symptoms-3,target_pcr_positive,2,2,17,6,35,468,35.530878,3.570402,0.458711,0.032319,0.739130,0.069583,526
8560,30138-COVID_Why-1,target_pcr_positive,2,2,36,54,16,420,17.067834,2.837196,0.458398,0.068441,0.400000,0.036697,526
14905,30141-covid_tst_symptoms-5,target_pcr_positive,2,2,21,13,31,461,23.332745,3.149858,0.456921,0.039924,0.617647,0.063008,526


In [31]:
# Cleaner table
pcr_results_clean = pcr_rows_sorted[
    [
        "concept_code",
        "target_concept_code",
        "n11",
        "odds_ratio",
        "phi",
        "p_target_given_code",
        "p_target_given_no_code"
    ]
].copy()

# Rename columns for readability
pcr_results_clean = pcr_results_clean.rename(columns={
    "concept_code": "Variable",
    "target_concept_code": "Outcome",
    "n11": "Both Present",
    "odds_ratio": "Odds Ratio",
    "phi": "Phi",
    "p_target_given_code": "P(PCR+ | Variable)",
    "p_target_given_no_code": "P(PCR+ | No Variable)"
})

# Round numeric values
numeric_cols = ["Odds Ratio", "Phi", "P(PCR+ | Variable)", "P(PCR+ | No Variable)"]
pcr_results_clean[numeric_cols] = pcr_results_clean[numeric_cols].round(3)

# Display top 10 cleaner results
pcr_results_clean.head(10)

,Variable,Outcome,Both Present,Odds Ratio,Phi,P(PCR+ | Variable),P(PCR+ | No Variable)
42408,31386-covid_results-1,target_pcr_positive,19,37.435,0.530,0.633,0.042
27125,30158-Symtpom_Neuro-7,target_pcr_positive,24,34.648,0.525,0.686,0.057
70982,target_athome_positive,target_pcr_positive,23,33.402,0.520,0.676,0.057
27360,30158-Symtpom_Neuro-8,target_pcr_positive,20,34.618,0.489,0.714,0.064
15140,30141-covid_tst_symptoms-6,target_pcr_positive,31,17.601,0.466,0.463,0.046
42416,31386-covid_results-2,target_pcr_positive,22,0.036,-0.464,0.049,0.593
21720,30153-Symptoms-6,target_pcr_positive,25,19.559,0.461,0.543,0.056
13965,30141-covid_tst_symptoms-3,target_pcr_positive,17,35.531,0.459,0.739,0.070
8560,30138-COVID_Why-1,target_pcr_positive,36,17.068,0.458,0.400,0.037
14905,30141-covid_tst_symptoms-5,target_pcr_positive,21,23.333,0.457,0.618,0.063


For readability, I created a simplified PCR results table with the most relevant columns: variable, outcome, number of records where both were present, odds ratio, phi coefficient, and conditional probabilities.

## Discussion

This project shows how COVID-19 risk can be estimated using information available before a clinic or emergency room visit. The DEMI knowledgebase helped organize relationships between symptoms, testing variables, and PCR-confirmed COVID-19. The temporal ordering was important because the model should only use information that would realistically be available before the PCR result.

Gradient Boosting performed best in this analysis. This may be because COVID-19 symptoms and home testing patterns are not always simple or linear. A more flexible model may be better able to capture combinations of symptoms and test-related variables.

There are several limitations. The dataset includes missing PCR results, and the number of positive cases is much smaller than the number of negative cases. Some variables in the knowledgebase may also be rare, which can make association measures look stronger than they really are. This model should not replace clinical judgment or diagnostic testing, but it demonstrates how a causal AI approach could support home-based risk prediction.

## References

References will include the course materials, DEMI algorithm resources, COVIDCARE project materials, and the assigned readings related to COVID-19 symptoms, symptom clusters, temporal order, and network models for diagnosis.